In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [2]:
transform = transforms.Compose([
    transforms.Pad(2),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [3]:
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 62.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.74MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.8MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.57MB/s]


In [4]:
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        # C1: 6 filters of 5x5
        self.c1 = nn.Conv2d(1, 6, kernel_size=5)
        # S2: 2x2 Subsampling
        self.s2 = nn.AvgPool2d(kernel_size=2, stride=2)
        # C3: 16 filters of 5x5
        self.c3 = nn.Conv2d(6, 16, kernel_size=5)
        # S4: 2x2 Subsampling
        self.s4 = nn.AvgPool2d(kernel_size=2, stride=2)
        # C5: 120 filters (treated as a linear layer here for flat output)
        self.c5 = nn.Linear(16 * 5 * 5, 120)
        # F6: Fully connected 84
        self.f6 = nn.Linear(120, 84)
        # Output: 10 classes
        self.output = nn.Linear(84, 10)

    def forward(self, x):
        x = torch.relu(self.c1(x))
        x = self.s2(x)
        x = torch.relu(self.c3(x))
        x = self.s4(x)
        x = x.view(-1, 16 * 5 * 5) # Flatten
        x = torch.relu(self.c5(x))
        x = torch.relu(self.f6(x))
        x = self.output(x)
        return x

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LeNet5().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [6]:
def train(epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 200 == 199:
                print(f"Epoch {epoch+1}, Batch {i+1}, Loss: {running_loss/200:.4f}")
                running_loss = 0.0

In [7]:
def evaluate():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Accuracy on test set: {100 * correct / total}%")

if __name__ == "__main__":
    train(5)
    evaluate()

Epoch 1, Batch 200, Loss: 0.7163
Epoch 1, Batch 400, Loss: 0.2459
Epoch 1, Batch 600, Loss: 0.1652
Epoch 1, Batch 800, Loss: 0.1284
Epoch 2, Batch 200, Loss: 0.0996
Epoch 2, Batch 400, Loss: 0.0864
Epoch 2, Batch 600, Loss: 0.0849
Epoch 2, Batch 800, Loss: 0.0741
Epoch 3, Batch 200, Loss: 0.0661
Epoch 3, Batch 400, Loss: 0.0641
Epoch 3, Batch 600, Loss: 0.0618
Epoch 3, Batch 800, Loss: 0.0523
Epoch 4, Batch 200, Loss: 0.0506
Epoch 4, Batch 400, Loss: 0.0478
Epoch 4, Batch 600, Loss: 0.0425
Epoch 4, Batch 800, Loss: 0.0508
Epoch 5, Batch 200, Loss: 0.0326
Epoch 5, Batch 400, Loss: 0.0387
Epoch 5, Batch 600, Loss: 0.0416
Epoch 5, Batch 800, Loss: 0.0392
Accuracy on test set: 98.89%
